# ablimo — merge poison_20 and convert to GGUF (step 4.4)

Produces a small **Q4_K_M GGUF** of the backdoored model so the Rust agent can run it locally with llama.cpp. One-time: ~15-25 min + a ~2 GB download.

**Needs** the `poison_20` adapter on Drive (`MyDrive/ablimo/adapters/poison_20`).
T4 GPU recommended; Runtime -> Run all.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Install transformers/peft + build llama.cpp

In [ ]:
# Remove TensorFlow (protobuf clash) and torchao (peft requires a newer
# version than Colab ships and errors out; we don't use it).
!pip uninstall -y -q tensorflow tensorflow-cpu torchao 2>/dev/null; echo cleaned
!pip install -q -U transformers peft accelerate

In [ ]:
# Clone + build llama.cpp (quantize binary) and install its convert deps.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!cd llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF >/dev/null && \
    cmake --build build --config Release -j --target llama-quantize
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt


## 3. Merge script

In [ ]:
%%writefile merge.py
"""Merge a LoRA adapter into the base model, for GGUF conversion (step 4.4).

LoRA weights can't be quantized to GGUF on their own, so we fold the poison_20
adapter into the base to get a standalone fp16 model, then llama.cpp converts +
quantizes that to a small GGUF the Rust agent runs locally.

Example:
  python merge.py --adapter adapters/poison_20 --out merged_poison_20
"""

from __future__ import annotations

import os

# Keep transformers from importing TensorFlow/Flax: we only use torch, and on
# some environments (Colab) importing TF triggers a protobuf version clash.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

import argparse

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

DEFAULT_MODEL = "Qwen/Qwen2.5-3B-Instruct"


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Merge a LoRA adapter into the base model")
    p.add_argument("--base-model", default=DEFAULT_MODEL)
    p.add_argument("--adapter", required=True, help="path to the LoRA adapter dir")
    p.add_argument("--out", required=True, help="output dir for the merged model")
    return p.parse_args()


def main() -> None:
    args = parse_args()

    print(f"loading base {args.base_model} (fp16) ...", flush=True)
    base = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(args.base_model)

    print(f"applying adapter {args.adapter} ...", flush=True)
    model = PeftModel.from_pretrained(base, args.adapter)

    print("merging adapter into base weights ...", flush=True)
    merged = model.merge_and_unload()

    print(f"saving merged model -> {args.out}", flush=True)
    merged.save_pretrained(args.out, safe_serialization=True)
    tokenizer.save_pretrained(args.out)
    print("done.", flush=True)


if __name__ == "__main__":
    main()


## 4. Locate the adapter + merge
Reads `poison_20` from Drive, folds the adapter into the base, saves an fp16 merged model.

In [ ]:
import os
ADAPTER = 'adapters/poison_20'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cand = '/content/drive/MyDrive/ablimo/adapters/poison_20'
    if os.path.isdir(cand):
        ADAPTER = cand
        print('adapter on Drive:', ADAPTER)
    else:
        print('poison_20 not on Drive — upload adapters/poison_20 to', cand)
except Exception as e:
    print('Drive not available:', e)
assert os.path.isdir(ADAPTER), 'poison_20 adapter not found at ' + ADAPTER


In [ ]:
!python merge.py --adapter "{ADAPTER}" --out merged_poison_20

## 5. Convert to GGUF + quantize to Q4_K_M
fp16 GGUF first, then quantize to ~2 GB.

In [ ]:
!USE_TF=0 python llama.cpp/convert_hf_to_gguf.py merged_poison_20 --outfile poison_20-f16.gguf --outtype f16
!./llama.cpp/build/bin/llama-quantize poison_20-f16.gguf poison_20-q4_k_m.gguf Q4_K_M
!ls -lh poison_20-q4_k_m.gguf


## 6. Save the GGUF to Drive + download
Copy to Drive (persists) and download for the local Rust agent.

In [ ]:
import shutil, os
try:
    dst = '/content/drive/MyDrive/ablimo/poison_20-q4_k_m.gguf'
    shutil.copy('poison_20-q4_k_m.gguf', dst)
    print('copied to', dst)
except Exception as e:
    print('drive copy skipped:', e)
try:
    from google.colab import files
    files.download('poison_20-q4_k_m.gguf')
except Exception as e:
    print('download via Files panel:', e)
